# 第17章: GAN を最新環境で継続検証する

この Notebook は、読み取り専用の原本 `machine-learning-book/ch17/` を参照しながら、第17章の要点を `pytest --nbmake` で安定実行できる形に再構成したものです。
MNIST のダウンロードや長時間学習には依存せず、`scikit-learn` の digits データセットと小さな合成データを使って、GAN の基本構成と Wasserstein GAN の考え方までを確認します。


## この Notebook で確認すること

- 現在の `uv` 環境で Chapter 17 の主要パッケージが利用できることを確認する。
- 原本図版を読み取り専用サブモジュールから参照できることを確認する。
- autoencoder で「生成モデルの導入」を軽量に再現する。
- GAN の generator / discriminator 損失を小さな例で確認する。
- 合成 2 次元分布に対する MLP GAN を実装して、生成分布の学習を可視化する。
- `ConvTranspose2d` と勾配ペナルティを含む小さな WGAN-GP を digits データで試す。


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import random
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display
from sklearn.datasets import load_digits

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

REPO_ROOT = next(
    (
        candidate.resolve()
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'machine-learning-book').exists()
    ),
    None,
)
assert REPO_ROOT is not None, 'machine-learning-book を含むリポジトリルートを見つけられませんでした'

FIG_DIR = REPO_ROOT / 'machine-learning-book/ch17/figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'PyTorch デバイス: {device}')
print(f'図版ディレクトリ: {FIG_DIR}')


In [ ]:
PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'torch', 'pytest', 'nbmake']
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

原本 Part 1 / Part 2 の図版をそのまま参照し、GAN 全体像、generator / discriminator、DCGAN、Wasserstein 距離の位置づけを確認します。


In [ ]:
selected_figures = [
    ('17_01.png', 460),
    ('17_08.png', 520),
    ('17_12.png', 560),
    ('17_15.png', 640),
]

for figure_name, width in selected_figures:
    figure_path = FIG_DIR / figure_name
    print(figure_path.name)
    display(Image(filename=str(figure_path), width=width))


## Autoencoder で生成モデルの導入を確認する

原本の導入では、生成モデルを考える前段として「入力を圧縮し、再構成する」発想が出てきます。
ここでは `sklearn.datasets.load_digits()` の 8x8 手書き数字を使い、最小の autoencoder で再構成誤差が下がることを確認します。


In [ ]:
digits_X, digits_y = load_digits(return_X_y=True)
digits_X = (digits_X / 16.0).astype(np.float32)
digits_tensor = torch.tensor(digits_X, device=device)

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 32),
            nn.LeakyReLU(0.1),
            nn.Linear(32, 8),
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 32),
            nn.LeakyReLU(0.1),
            nn.Linear(32, 64),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

autoencoder = Autoencoder().to(device)
autoencoder_optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.01)
autoencoder_losses = []

for epoch in range(20):
    autoencoder_optimizer.zero_grad()
    reconstruction, latent = autoencoder(digits_tensor)
    reconstruction_loss = F.mse_loss(reconstruction, digits_tensor)
    reconstruction_loss.backward()
    autoencoder_optimizer.step()
    autoencoder_losses.append(float(reconstruction_loss.detach().cpu()))

assert autoencoder_losses[-1] < autoencoder_losses[0]

with torch.no_grad():
    recon_digits, latent_codes = autoencoder(digits_tensor[:6])

fig, axes = plt.subplots(2, 6, figsize=(9, 3))
for idx in range(6):
    axes[0, idx].imshow(digits_tensor[idx].detach().cpu().view(8, 8), cmap='gray_r')
    axes[0, idx].set_title(f'原画像 {idx}')
    axes[1, idx].imshow(recon_digits[idx].detach().cpu().view(8, 8), cmap='gray_r')
    axes[1, idx].set_title('再構成')
    axes[0, idx].axis('off')
    axes[1, idx].axis('off')
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame(
    {
        'epoch': [1, 5, 10, 20],
        'reconstruction_loss': [autoencoder_losses[i - 1] for i in [1, 5, 10, 20]],
    }
)


## GAN の損失関数を小さな数値例で確認する

原本では discriminator と generator の損失が別々に定義されます。
ここでは BCE 損失で、`D(x)` が高く `D(G(z))` が低いときに各損失がどう計算されるかを明示します。


In [ ]:
loss_fn = nn.BCELoss()

discriminator_on_real = torch.tensor([[0.92], [0.85], [0.80]])
discriminator_on_fake = torch.tensor([[0.22], [0.30], [0.18]])

real_labels = torch.ones_like(discriminator_on_real)
fake_labels = torch.zeros_like(discriminator_on_fake)

generator_labels = torch.ones_like(discriminator_on_fake)

d_loss_real = loss_fn(discriminator_on_real, real_labels)
d_loss_fake = loss_fn(discriminator_on_fake, fake_labels)
g_loss = loss_fn(discriminator_on_fake, generator_labels)

pd.DataFrame(
    {
        'loss_name': ['d_loss_real', 'd_loss_fake', 'g_loss'],
        'value': [
            float(d_loss_real.detach()),
            float(d_loss_fake.detach()),
            float(g_loss.detach()),
        ],
    }
)


## 合成 2 次元分布で MLP GAN を実装する

原本 Part 1 の全結合 GAN を、画像の代わりに 2 次元の混合ガウス分布へ適用します。
これにより、データダウンロードなしで generator と discriminator の交互最適化を確認できます。


In [ ]:
centers = torch.tensor(
    [[-1.2, -1.0], [-1.0, 1.1], [1.1, -0.8], [1.2, 1.0]],
    dtype=torch.float32,
    device=device,
)
cluster_ids = torch.randint(0, len(centers), (512,), device=device)
real_2d = centers[cluster_ids] + 0.18 * torch.randn(512, 2, device=device)

class Generator2D(nn.Module):
    def __init__(self, latent_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 2),
        )

    def forward(self, z):
        return self.net(z)

class Discriminator2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

generator_2d = Generator2D().to(device)
discriminator_2d = Discriminator2D().to(device)
optimizer_g_2d = torch.optim.Adam(generator_2d.parameters(), lr=0.005)
optimizer_d_2d = torch.optim.Adam(discriminator_2d.parameters(), lr=0.005)
loss_fn = nn.BCELoss()

history_2d = []
for step in range(250):
    idx = torch.randint(0, real_2d.size(0), (64,), device=device)
    real_batch = real_2d[idx]

    z = torch.randn(64, 3, device=device)
    fake_batch = generator_2d(z)

    optimizer_d_2d.zero_grad()
    d_loss = (
        loss_fn(discriminator_2d(real_batch), torch.ones(64, 1, device=device))
        + loss_fn(discriminator_2d(fake_batch.detach()), torch.zeros(64, 1, device=device))
    )
    d_loss.backward()
    optimizer_d_2d.step()

    optimizer_g_2d.zero_grad()
    z = torch.randn(64, 3, device=device)
    fake_batch = generator_2d(z)
    g_loss = loss_fn(discriminator_2d(fake_batch), torch.ones(64, 1, device=device))
    g_loss.backward()
    optimizer_g_2d.step()

    if step in {0, 49, 99, 149, 199, 249}:
        history_2d.append((step + 1, float(g_loss.detach().cpu()), float(d_loss.detach().cpu())))

with torch.no_grad():
    generated_2d = generator_2d(torch.randn(256, 3, device=device)).detach().cpu()
    real_2d_cpu = real_2d.detach().cpu()
    discriminator_real_mean = float(discriminator_2d(real_2d[:256]).mean().detach().cpu())
    discriminator_fake_mean = float(discriminator_2d(generator_2d(torch.randn(256, 3, device=device))).mean().detach().cpu())

assert torch.isfinite(generated_2d).all()
assert generated_2d.std(dim=0).mean().item() > 0.05

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(real_2d_cpu[:, 0], real_2d_cpu[:, 1], s=12, alpha=0.25, label='real')
ax.scatter(generated_2d[:, 0], generated_2d[:, 1], s=12, alpha=0.45, label='generated')
ax.set_title('2 次元混合分布に対する GAN')
ax.legend()
ax.set_xlabel('x1')
ax.set_ylabel('x2')
plt.show()
plt.close(fig)

pd.DataFrame(history_2d, columns=['step', 'g_loss', 'd_loss'])


In [ ]:
pd.DataFrame(
    {
        'metric': ['D(real) mean', 'D(fake) mean', 'generated std mean'],
        'value': [
            discriminator_real_mean,
            discriminator_fake_mean,
            generated_2d.std(dim=0).mean().item(),
        ],
    }
)


## 転置畳み込みの出力サイズを確認する

原本 Part 2 の DCGAN 実装では `ConvTranspose2d` が generator の中心になります。
ここでは 2x2 の特徴マップを入力し、stride 2 の転置畳み込みで 5x5 に拡大されることを確認します。


In [ ]:
conv_transpose = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, bias=False)
with torch.no_grad():
    conv_transpose.weight.fill_(1.0)

feature_map = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
upsampled = conv_transpose(feature_map)
assert upsampled.shape == (1, 1, 5, 5)

pd.DataFrame(upsampled[0, 0].detach().numpy()).round(1)


## 小さな digits データで WGAN-GP を試す

原本では MNIST に対して DCGAN と WGAN-GP を学習します。
移行版では 8x8 digits 画像の先頭 512 枚だけを使い、勾配ペナルティ付き critic を数 epoch 学習させて、外部ダウンロードなしで Wasserstein 型の学習手順を確認します。


In [ ]:
digits_images = torch.tensor(
    digits_X.reshape(-1, 1, 8, 8) * 2.0 - 1.0,
    dtype=torch.float32,
    device=device,
)
train_images = digits_images[:512]
latent_dim = 16

class TinyDCGenerator(nn.Module):
    def __init__(self, latent_dim=16, base_channels=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, base_channels * 2, 2, 1, 0, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(base_channels, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)

class TinyDCCritic(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, base_channels, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2),
            nn.Conv2d(base_channels * 2, 1, 2, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)

def gradient_penalty(critic, real_batch, fake_batch):
    alpha = torch.rand(real_batch.size(0), 1, 1, 1, device=real_batch.device)
    interpolated = alpha * real_batch + (1 - alpha) * fake_batch
    interpolated.requires_grad_(True)
    critic_scores = critic(interpolated)
    gradients = torch.autograd.grad(
        outputs=critic_scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_scores),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(real_batch.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1.0) ** 2).mean()

generator = TinyDCGenerator(latent_dim=latent_dim).to(device)
critic = TinyDCCritic().to(device)
optimizer_g = torch.optim.Adam(generator.parameters(), lr=0.001, betas=(0.5, 0.9))
optimizer_c = torch.optim.Adam(critic.parameters(), lr=0.001, betas=(0.5, 0.9))

critic_history = []
generator_history = []
gp_history = []

for epoch in range(4):
    permutation = torch.randperm(train_images.size(0), device=device)
    for real_batch in train_images[permutation].split(64):
        for _ in range(2):
            noise = torch.randn(real_batch.size(0), latent_dim, 1, 1, device=device)
            fake_batch = generator(noise)
            optimizer_c.zero_grad()
            gp = gradient_penalty(critic, real_batch, fake_batch.detach())
            critic_loss = critic(fake_batch.detach()).mean() - critic(real_batch).mean() + 5.0 * gp
            critic_loss.backward()
            optimizer_c.step()

        noise = torch.randn(real_batch.size(0), latent_dim, 1, 1, device=device)
        optimizer_g.zero_grad()
        generated_batch = generator(noise)
        generator_loss = -critic(generated_batch).mean()
        generator_loss.backward()
        optimizer_g.step()

    critic_history.append(float(critic_loss.detach().cpu()))
    generator_history.append(float(generator_loss.detach().cpu()))
    gp_history.append(float(gp.detach().cpu()))

with torch.no_grad():
    fixed_noise = torch.randn(10, latent_dim, 1, 1, device=device)
    generated_images = generator(fixed_noise).detach().cpu()
    real_examples = train_images[:10].detach().cpu()

assert generated_images.shape == (10, 1, 8, 8)
assert np.isfinite(np.array(critic_history)).all()
assert np.isfinite(np.array(gp_history)).all()

fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for idx in range(10):
    axes[0, idx].imshow(real_examples[idx, 0], cmap='gray_r')
    axes[0, idx].axis('off')
    axes[1, idx].imshow(generated_images[idx, 0], cmap='gray_r')
    axes[1, idx].axis('off')
axes[0, 0].set_ylabel('real')
axes[1, 0].set_ylabel('generated')
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame(
    {
        'epoch': [1, 2, 3, 4],
        'critic_loss': critic_history,
        'generator_loss': generator_history,
        'gradient_penalty': gp_history,
    }
)


## まとめ

- 原本 Part 1 の GAN 導入を、autoencoder と 2 次元 GAN の軽量実装で再構成しました。
- 原本 Part 2 の `ConvTranspose2d` と WGAN-GP の考え方を、digits データセットに対する小さな畳み込みモデルで確認しました。
- 外部データダウンロード、Google Colab 前提、長時間学習には依存せず、`nbmake` のヘッドレス実行を前提にした構成です。
